# 00 Problem Definition

## What is Slay the Spire?

Slay the Spire is a roguelike deck-building game. Players attempt to ascendthe spire (map) across several "Acts," fighting monsters in turn-based cardbattles. Winning fights rewards a choice of new cards to add to your deck: 3 options of differing rarities are offered, drawn randomly from a card pool,and the player can pick one (or skip). These cards permanently shape the deckfor the rest of the run. A run ends in either victory (clearing all Acts)or death; "Ascension levels" add optional difficulty modifiers that makeruns progressively harder.

<div align="center">

![Card reward screen](images/card-reward.png)

</div>

## 1. Research Question

On the card reward screen, given the choice of cards offered, which selections improve (or worsen) run outcomes? How do these effects vary by character or ascension level?

## 2. Motivation for Study

Slay the Spire's card reward screen offers a small set of cardsat various points in a run. Some picks are widely considered strong or weakby the community at large (forums, Reddit, among others). But this is rarelyquantified directly against actual aggregated run outcomes. This projectmeasures that directly: for each pick, what's the average "floors gained"after that point in the run, and does the answer change with character orascension level. My hypothesis is that at higher ascension, strong/weak picks mayseparate more analogous to how move quality matters more at higher chessskill levels. As the margin for error in move chess gets lower, the optimal choices converge towards a "single best move". In STS, sub-optimal playsyou could get away with at lower ascension levels because you don't need everyhealth point possible to win the final bosses. Health point management becomes a necessity at the higher ascensions though, which means taking the best cardspossible is required because you need to mitigate HP loss at earlier fights more.I have been recently playing a lot of Slay The Spire 1/2, both solo and with friends,and I struggle with high ascension winning and knowing what cards are good to pick.I wanted to get statistical proof for what the optimal card choices so it would help meget past the higher ascension levels I haven't beat yet.

## 3. Data

The source for the data comes from Jake Rabinowitz; he handled the Slay the Spire internal metrics tools and created backups of STS run data over the course of 2 years [Link to Reddit Post](https://www.reddit.com/r/slaythespire/comments/jt5y1w/77_million_runs_an_sts_metrics_dump/). He was able to compile 77 million runs over that timeframe, chunked across 52,000 files that contain about 14,000 runs each. This totaled 65GB overall, which I'm not willing to dedicate that much computer storage space to, and the sample size is way more than I need for a personal analysis project. Instead, I went ahead and used dropbox sample. This contains ~124,000 runs from November 2020 at about 68MB storage required, which is perfect to get started with.

Source: Jake Rabinowitz's community Slay the Spire run dataset (~120K-run sample used initially; full dataset is ~77M runs). Sample stored as `.json` is stored as a `.json.gz` file with (confirmed) fields including:

- `card_choices`: list of {`picked`, `not_picked`, `floor`} events
- `character_chosen`, `ascension_level`
- `victory`, `floor_reached`, `score`
- `current_hp_per_floor`, `gold_per_floor`, `max_hp_per_floor`
- `master_deck`, `relics`, `relics_obtained`, `path_taken`, `path_per_floor`
- `is_endless`, `is_daily`, `is_trial`, `chose_seed`

#### Known issue: abandoned runs
An initial pass found an unfiltered win rate (`victory`) of ~0.18%, which is implausibly low. Likely cause: the raw dataset includes many runs abandoned early (Act 1 deaths, seed re-rolls, etc.) that never represented meaningful play. These runs deflate outcome measures and leave too few `card_choices` events to be useful. Filtering these out is necessary before outcome measures are meaningful; likely will selected a floor to the "floor_reached" 

## 4. Filtering Plan

To control for practiced/rehearsed runs and to reduce the effect of abandoned runs on analysis, the following are planned to be used for data filtering. This is not an exhaustive list, and more (or less) may be used as further data exploration is done.

- `is_endless == False`
- `is_daily == False`
- `is_trial == False`
- `chose_seed == False`
- `floor_reached >= [5–15, TBD]`

## 5. Outcome Metric

There are a few possible options for what a successful run could mean:

- `victory`: whether Act 3 was beaten
- `floor_reached`: The highest floor reached on the run
- `floors_gained`: a created variable noted below
- All of the above

For each `card_choices` event at floor F in a run that reaches `floor_reached`, define:

    floors_gained = floor_reached - F

This gives a per-pick outcome: how far the run continued after that particular reward decision.

#### Baseline / comparison
Raw `floors_gained` by card isn't directly comparable across picks unless we account for when the pick happened (a card picked at floor 3 has more room to show a large `floors_gained` than one picked at floor 45) and what else was offered (`not_picked`). Planned baseline: compare floors_gained for `picked` vs. the average of `not_picked` cards in the same choice set while controlling for floor. This gives a relative +/- vs. alternatives
number rather than just a raw average per card.

## 6. Confounders / Covariates to Consider

- `Floor` at time of pick: room left to gain floors mechanically shrinks
- `current_hp_per_floor` / `max_hp_per_floor` at time of pick
- `ascension_level`
- `character_chosen`
- `len(master_deck)`: a card added to a 15-card deck vs. a 30-card deck has more inherent impact
- `relics`: some relics meaningfully change a card's value, e.g. relics that scale with card type or block/attack count
- `gold` / `gold_per_floor` at time of pick: affects ability to shop for removals/relics, indirectly shaping deck quality
- `path_taken` / `path_per_floor`: route choice affects enemy/elite exposure independent of card picks
- `items_purged` / `purchased_purges` at time of pick: deck thinning changes what a newly added card's relative impact looks like
- `neow_bonus`: starting bonus/penalty alters baseline state before any reward picks happen
- `build_version`: card balance may differ across versions in the sample

## 7. Next Steps

1. Build ETL pipeline to collect and curate the full dataset needed for analysis, split across two notebooks:
   - **Data Collection**: extract and parse raw `.json` run file(s) into readable Parquet format; this will be the raw file
   - **Data Preparation**: apply run-level filters (`is_endless`, `is_daily`, `is_trial`, `chose_seed` floor_reached threshold, etc.), develop `card_choices` into a per-pick-event table, select relevant fields/covariates, handle     missing or malformed records, compute `floors_gained`, and write the cleaned dataset to Parquet for analysis.
2. Recompute win rate / `floor_reached` distribution post-preparation.
3. Explore baseline-adjusted `floors_gained` comparisons once the dataset is built.